# 00: 環境と観測の確認

ロボットの状態を取得し、指令の値を1つ変えると手先がどう動くかを、動画と数値で見比べます。依存ライブラリは実行時に取得します。

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `00_environment` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `lift.json`
- `lift_tools.py`
- `prepare.py`
- `requirements.txt`

In [ ]:
# @title この回の配布ファイルを準備する
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/00_environment"  # @param {type:"string"}
REQUIRED_FILES = [
    "lift.json",
    "lift_tools.py",
    "prepare.py",
    "requirements.txt",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_00")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## 環境を準備する

Liftは、ロボットアームで立方体を持ち上げる練習用の環境です。下のセルで、Liftを起動する処理と、リセット直後にシミュレーションを進める処理を用意します。

### stateとactionの関係

| 変数 | 何を表すか | この実習での内訳 |
| --- | --- | --- |
| `state`（8個） | 観測した手先とグリッパーの状態 | 手先の位置3個、向き3個、左右の指の関節位置2個 |
| `action`（7個） | シミュレータに送る次の指令 | 手先の移動3個、回転3個、グリッパーを開閉する指令1個 |

`obs` にはカメラ画像とロボットの観測値が入っています。`state_vector(obs)` は、その観測値から手先と指の状態を8個の数値にまとめます。この回はあらかじめ決めた `action` を `env.step(action)` に渡し、シミュレーションを1ステップ進めて、次の `obs` を受け取ります。ログの `state_before` は、その行の `action` を送る直前の状態です。

### 記録前にシミュレーションを進める

`env.reset()` の直後は、物体やロボットの初期配置が落ち着くまで少し動くことがあります。そこで、手先の移動・回転をゼロにしてグリッパーを開く指令を `settle_steps` 回送り、その後の観測を取得します。初期設定は10ステップで、1秒あたり20ステップ進むので、シミュレーション内の0.5秒に相当します。`obs, _, _, _` は、`env.step()` が返す観測・報酬・終了フラグ・補足情報のうち、新しい観測だけを使う書き方です。残りは `_` で受け取ります。

In [ ]:
LIFT_CODE = r"""
import json
import os
from pathlib import Path

# 描画バックエンドをrobosuiteのimportより先に設定します。
settings = json.loads(Path("lift.json").read_text())
config = settings["env"]
os.environ["MUJOCO_GL"] = config["renderer"]
os.environ["PYOPENGL_PLATFORM"] = config["renderer"]
import robosuite
from robosuite import macros
from lift_tools import lift_controller, state_vector

macros.IMAGE_CONVENTION = "opengl"
STATE_ORDER = ["x", "y", "z", "rx", "ry", "rz", "gripper_left", "gripper_right"]
ACTION_ORDER = ["dx", "dy", "dz", "drx", "dry", "drz", "gripper"]


def make_lift():
    return robosuite.make(
        env_name=config["task"],
        robots=config["robot"],
        controller_configs=lift_controller(),
        has_renderer=False,
        has_offscreen_renderer=True,
        use_camera_obs=True,
        use_object_obs=False,
        camera_names=config["cameras"],
        camera_heights=config["image_size"],
        camera_widths=config["image_size"],
        control_freq=config["control_freq"],
        horizon=settings["max_steps"] + config["settle_steps"],
        hard_reset=False,
        reward_shaping=False,
        seed=settings["seed"],
    )


def reset_and_settle(env):
    obs = env.reset()
    for _ in range(config["settle_steps"]):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
    return obs
"""

## 実際にstateを取得する

このセルを実行すると、リセット後の観測から `state_vector(obs)` で8個の値を取得し、項目名と一緒に表示します。`x, y, z` は手先の位置（m）、`rx, ry, rz` は手先の向きを表す回転ベクトル（rad）、最後の2つは左右の指の関節位置（m）です。

In [ ]:
run_python(LIFT_CODE + r"""
env = make_lift()
try:
    obs = reset_and_settle(env)
    state = state_vector(obs)
    print("初期配置が落ち着いた後のstate（8個）")
    for name, value in zip(STATE_ORDER, state, strict=True):
        print(f"{name:>13s}  {value:+.6f}")
finally:
    env.close()
""")

## actionの先頭だけ変えて試す

基準の `[0, 0, 0, 0, 0, 0, -1]` では、手先の移動と回転の指令をゼロにして、グリッパーを開きます。最初の3個が移動、次の3個が回転、最後の1個が開閉です。Pandaのグリッパーでは `-1` で開き、`+1` で閉じます。

比較用は、先頭の `dx` だけを `0.25` に変えます。これは世界座標のx軸の正の方向へ手先を動かす指令です。移動・回転の値は `-1` から `1` の範囲で指定し、コントローラが移動量・回転量に換算します。`np.array(..., dtype=np.float32)` は、指令の値を32ビット浮動小数点数のNumPy配列にします。

両方を同じ初期条件から始め、同じステップ数だけ進めます。次のセルでは、手先が動く様子と、開始時から最後までの `state` の変化を比べられます。

[robosuiteのコントローラ](https://robosuite.ai/docs/modules/controllers.html) · [Pandaのグリッパーの実装](https://github.com/ARISE-Initiative/robosuite/blob/95743f6687ad7394ccb6e865fa4e1b99114e580d/robosuite/models/grippers/panda_gripper.py#L35-L54)

In [ ]:
result = run_python(LIFT_CODE + r"""
import tempfile

import imageio.v2 as imageio
import numpy as np

base_action = np.array([0, 0, 0, 0, 0, 0, -1], dtype=np.float32)
changed_action = base_action.copy()
changed_action[0] = 0.25  # 先頭の dx だけを変えます。

camera_key = f'{config["cameras"][0]}_image'
trials = []
clips = []
for action in (base_action, changed_action):
    env = make_lift()
    try:
        obs = reset_and_settle(env)
        initial_state = state_vector(obs)
        frames = [obs[camera_key][::-1].copy()]
        steps = []
        for step in range(settings["max_steps"]):
            state_before = state_vector(obs)
            obs, reward, done, _ = env.step(action)
            state_after = state_vector(obs)
            frames.append(obs[camera_key][::-1].copy())
            steps.append({
                "step": step,
                "state_before": state_before.tolist(),
                "action": action.tolist(),
                "state_after": state_after.tolist(),
                "reward": reward,
            })
            if done:
                break
        trials.append({
            "action": action.tolist(),
            "initial_state": initial_state.tolist(),
            "final_state": state_vector(obs).tolist(),
            "steps": steps,
        })
        clips.append(frames)
    finally:
        env.close()

np.testing.assert_allclose(
    trials[0]["initial_state"], trials[1]["initial_state"], atol=1e-6, rtol=0
)
folder = Path(tempfile.mkdtemp(prefix="lift-", dir="outputs")).resolve()
imageio.imwrite(folder / "observation.png", clips[0][0])
with imageio.get_writer(folder / "video.mp4", fps=config["control_freq"]) as video:
    for left, right in zip(*clips, strict=True):
        video.append_data(np.concatenate((left, right), axis=1))
report = {
    "camera_shape": list(clips[0][0].shape),
    "state_order": STATE_ORDER,
    "action_order": ACTION_ORDER,
    "trials": trials,
}
(folder / "observations.json").write_text(json.dumps(report, indent=2))
print(f"Results: {folder}")
""")

## 動画とstateの変化を見比べる

動画の左は基準、右は `dx` を変えたときの動きです。動画の下に出る表の「差」は、最後の値から開始時の値を引いたものです。まず `x` の差を見てください。`dx` は指令の値で、表の `x` はシミュレータで観測した位置（m）です。

In [ ]:
import json

from IPython.display import Image, Markdown, Video, display

report = json.loads((result / "observations.json").read_text())
baseline, changed = report["trials"]
print("基準のaction（左）:", baseline["action"])
print("比較のaction（右）:", changed["action"])
print("カメラ画像の形:", report["camera_shape"])
print("開始時のカメラ画像")
display(Image(filename=str(result / "observation.png")))
display(Video(str(result / "video.mp4"), embed=True))

rows = [
    "| 項目 | 開始時 | 基準の最後 | 基準の差 | 比較の最後 | 比較の差 |",
    "| --- | ---: | ---: | ---: | ---: | ---: |",
]
for name, start, left, right in zip(
    report["state_order"], baseline["initial_state"],
    baseline["final_state"], changed["final_state"], strict=True,
):
    rows.append(
        f"| `{name}` | {start:+.5f} | {left:+.5f} | {left - start:+.5f} "
        f"| {right:+.5f} | {right - start:+.5f} |"
    )
display(Markdown("\n".join(rows)))

## 確認課題

右の手先はどちらに動き、`x` はどれだけ変わりましたか。`changed_action[0]` を `-0.25` にして比較のセルと表示のセルを再実行すると、動画と表はどう変わるか確かめてください。

## 結果をDriveに保存する

開始時の画像、比較動画、両方の試行の状態・行動ログを保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "00_environment_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)